# SMS Spam Detection

CodSoft ML Internship Task using your uploaded `spam.csv`. This notebook reproduces training/evaluation and custom predictions.

In [ ]:
from pathlib import Path
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'data'/'spam.csv').exists() and (ROOT.parent/'data'/'spam.csv').exists(): ROOT=ROOT.parent
df_raw=pd.read_csv(ROOT/'data'/'spam.csv',encoding='latin-1')
df_raw.head()

In [ ]:
df=df_raw[['v1','v2']].rename(columns={'v1':'label','v2':'message'}).dropna().copy()
df['label']=df.label.astype(str).str.strip().str.lower(); df['message']=df.message.astype(str).str.strip()
df=df[df.label.isin(['ham','spam']) & df.message.ne('')].drop_duplicates().reset_index(drop=True)
print(df.shape); print(df.label.value_counts())

In [ ]:
import matplotlib.pyplot as plt
df.label.value_counts().plot(kind='bar',title='SMS label distribution',rot=0); plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df.message,df.label,test_size=.2,random_state=42,stratify=df.label)
print('Train:',len(X_train),'Test:',len(X_test))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
models={'Multinomial Naive Bayes':Pipeline([('tfidf',TfidfVectorizer(stop_words='english',ngram_range=(1,2))),('classifier',MultinomialNB())]),'Logistic Regression':Pipeline([('tfidf',TfidfVectorizer(stop_words='english',ngram_range=(1,2))),('classifier',LogisticRegression(max_iter=2000))])}
for m in models.values(): m.fit(X_train,y_train)

In [ ]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report
rows=[]
for name,m in models.items():
 p=m.predict(X_test); rows.append({'Model':name,'Accuracy':accuracy_score(y_test,p),'Spam Precision':precision_score(y_test,p,pos_label='spam'),'Spam Recall':recall_score(y_test,p,pos_label='spam'),'Spam F1':f1_score(y_test,p,pos_label='spam')})
results=pd.DataFrame(rows).sort_values('Spam F1',ascending=False); results

In [ ]:
best_name=results.iloc[0]['Model']; best_model=models[best_name]
print('Selected:',best_name); print(classification_report(y_test,best_model.predict(X_test)))
cm=confusion_matrix(y_test,best_model.predict(X_test),labels=['ham','spam']); print('Confusion matrix (actual rows, predicted columns):\n',cm)

In [ ]:
plt.figure(figsize=(5,4)); plt.imshow(cm); plt.xticks([0,1],['ham','spam']); plt.yticks([0,1],['ham','spam']); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix');
for i in range(2):
 for j in range(2): plt.text(j,i,cm[i,j],ha='center',va='center')
plt.show()

In [ ]:
import joblib
(ROOT/'models').mkdir(exist_ok=True)
joblib.dump(best_model,ROOT/'models'/'best_sms_spam_model.joblib')
print('Saved model')

In [ ]:
def predict_sms(message):
 pred=best_model.predict([message])[0]; prob=best_model.predict_proba([message])[0]; confidence=prob[list(best_model.classes_).index(pred)]; return pred,float(confidence)
for msg in ['Congratulations! You have won a free prize. Call now to claim.','Hey, are we meeting at 6 today?']:
 print(msg,'=>',predict_sms(msg))